### A. Implementing attention mechanisms in an NLP model using TensorFlow/Keras.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

#  Hyperparameters (tweak for your task) 
SRC_VOCAB = 8000
TGT_VOCAB = 8000
EMB_DIM    = 256
UNITS      = 256         # GRU hidden size
SRC_MAXLEN = 50
TGT_MAXLEN = 50

#  Encoder 
encoder_inputs = layers.Input(shape=(SRC_MAXLEN,), name="encoder_inputs")
enc_emb = layers.Embedding(SRC_VOCAB, EMB_DIM, mask_zero=True, name="enc_embedding")(encoder_inputs)
enc_out, enc_state = layers.GRU(UNITS, return_sequences=True, return_state=True, name="enc_gru")(enc_emb)
# enc_out: (B, SrcLen, UNITS)  | enc_state: (B, UNITS)

#  Decoder (teacher forcing) 
decoder_inputs = layers.Input(shape=(TGT_MAXLEN,), name="decoder_inputs")
dec_emb = layers.Embedding(TGT_VOCAB, EMB_DIM, mask_zero=True, name="dec_embedding")(decoder_inputs)
dec_seq, _ = layers.GRU(UNITS, return_sequences=True, return_state=True, name="dec_gru")(dec_emb, initial_state=enc_state)
# dec_seq: (B, TgtLen, UNITS)  — this is the query for attention at each step

#  Bahdanau / Additive Attention 
# Query: dec_seq (B, Tq, UNITS)
# Value/Key: enc_out (B, Tv, UNITS)
attn = layers.AdditiveAttention(name="bahdanau_attention")
context = attn([dec_seq, enc_out])              # (B, Tq, UNITS)
# Optional: expose attention weights via a custom call if needed in an interview.

#  Concatenate context with decoder sequence and predict tokens 
x = layers.Concatenate(name="concat_context")([dec_seq, context])   # (B, Tq, 2*UNITS)
x = layers.TimeDistributed(layers.Dense(UNITS, activation="tanh"), name="fusion_dense")(x)
logits = layers.TimeDistributed(layers.Dense(TGT_VOCAB), name="token_logits")(x)  # from_logits=True

model = Model([encoder_inputs, decoder_inputs], logits, name="seq2seq_bahdanau")
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

model.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

#  Hyperparameters 
SRC_VOCAB = 8000
TGT_VOCAB = 8000
EMB_DIM    = 256
UNITS      = 256         # GRU hidden size
SRC_MAXLEN = 50
TGT_MAXLEN = 50

# Encoder
encoder_inputs = layers.Input(shape=(SRC_MAXLEN,), name="encoder_inputs")
enc_emb = layers.Embedding(SRC_VOCAB, EMB_DIM, mask_zero=True, name="enc_embedding")(encoder_inputs)
enc_out, enc_state = layers.GRU(UNITS, return_sequences=True, return_state=True, name="enc_gru")(enc_emb)
# enc_out: (B, SrcLen, UNITS)  | enc_state: (B, UNITS)

# Decoder 
decoder_inputs = layers.Input(shape=(TGT_MAXLEN,), name="decoder_inputs")
dec_emb = layers.Embedding(TGT_VOCAB, EMB_DIM, mask_zero=True, name="dec_embedding")(decoder_inputs)
dec_seq, _ = layers.GRU(UNITS, return_sequences=True, return_state=True, name="dec_gru")(dec_emb, initial_state=enc_state)
# dec_seq: (B, TgtLen, UNITS)  — this is the query for attention at each step

# Bahdanau / Additive Attention
# Query: dec_seq (B, Tq, UNITS)
# Value/Key: enc_out (B, Tv, UNITS)
attn = layers.AdditiveAttention(name="bahdanau_attention")
context = attn([dec_seq, enc_out])              # (B, Tq, UNITS)
# Optional: expose attention weights via a custom call if needed in an interview.

#  Concatenate context with decoder sequence and predict tokens 
x = layers.Concatenate(name="concat_context")([dec_seq, context])   # (B, Tq, 2*UNITS)
x = layers.TimeDistributed(layers.Dense(UNITS, activation="tanh"), name="fusion_dense")(x)
logits = layers.TimeDistributed(layers.Dense(TGT_VOCAB), name="token_logits")(x)  # from_logits=True

model = Model([encoder_inputs, decoder_inputs], logits, name="seq2seq_bahdanau")
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

model.summary()

In [ ]:
class LuongAttention(layers.Layer):
    """
    attn_type: "dot" or "general"
    query:  (batch, d_q)        -> decoder hidden state
    values: (batch, T, d_v)     -> encoder outputs
    mask:   (batch, T) or None
    returns: context (batch, d_v), attn_weights (batch, T)
    """
    def __init__(self, attn_type="dot"):
        super().__init__()
        assert attn_type in ("dot", "general")
        self.attn_type = attn_type
        self.Wa = None  # created lazily for "general"

    def build(self, input_shapes):
        # no explicit build from Keras call signature, so defer to first call
        pass

    def _ensure_Wa(self, values, query):
        if self.attn_type == "general" and self.Wa is None:
            d_q = query.shape[-1]
            self.Wa = layers.Dense(d_q, use_bias=False)  # project values to query dim

    def call(self, query, values, mask=None):
        # query: (batch, d_q), values: (batch, T, d_v)
        self._ensure_Wa(values, query)

        if self.attn_type == "dot":
            # scores_t = v_t · q
            scores = tf.reduce_sum(values * tf.expand_dims(query, 1), axis=-1)      # (batch, T)
        else:  # "general"
            v_proj = self.Wa(values)                                                # (batch, T, d_q)
            scores = tf.reduce_sum(v_proj * tf.expand_dims(query, 1), axis=-1)      # (batch, T)

        if mask is not None:
            minus_inf = tf.constant(-1e9, dtype=scores.dtype)
            scores = tf.where(tf.equal(mask, 1), scores, minus_inf)

        attn_weights = tf.nn.softmax(scores, axis=-1)                                # (batch, T)
        context = tf.reduce_sum(tf.expand_dims(attn_weights, -1) * values, axis=1)   # (batch, d_v)
        return context, attn_weights

### B. Practical session: Building a simple Transformer model from scratch.

In [3]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 

# Hyperparameters 
VOCAB_SIZE = 20000
MAX_LEN = 200
EMBED_DIM = 64 
NUM_HEADS = 4
FF_DIM = 128 
NUM_ENCODER_BLOCKS = 2 
DROPOUT = 0.1
BATCH_SIZE = 256
EPOCHS = 1 

# Data 
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=MAX_LEN, padding='post', truncating='post')

# Positional Embedding Layer 
class PositionalEmbedding(layers.Layer):
    def __init__(self, vocab_size, embed_dim, max_len, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(max_len, embed_dim)

    def compute_mask(self, inputs, mask=None):
        return self.token_emb.compute_mask(inputs)
    
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions 

In [4]:
# Transformer Encoder Block
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        attn_mask = None 
        if mask is not None:
            attn_mask = tf.cast(mask[:, tf.newaxis, tf.newaxis, :],dtype=tf.bool)

        x1 = self.norm1(x)
        attn_out = self.attn(x1, x1, attention_mask=attn_mask, training=training)
        attn_out = self.drop1(attn_out, training = training )
        x2 = x+attn_out

        y1 = self.norm2(x2)
        ffn_out = self.ffn(y1)
        ffn_out = self.drop2(ffn_out, training = training)
        return x2 + ffn_out 

In [7]:
inputs = keras.Input(shape=(MAX_LEN, ), dtype='int32')
x = PositionalEmbedding(VOCAB_SIZE, EMBED_DIM, MAX_LEN)(inputs)
mask = x._keras_mask 

for _ in range(NUM_ENCODER_BLOCKS):
    x = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT)(x, mask=mask)

class MaskedWhere(layers.Layer):
    def call(self, inputs):
        x, mask, neg_inf = inputs
        mask_expanded = tf.expand_dims(mask, -1)
        return tf.where(mask_expanded, x, neg_inf)
    
if mask is not None:
    neg_inf = tf.cast(-1e9, x.dtype)
    masked_x = MaskedWhere()([x, mask, neg_inf])
    x = tf.reduce_max(masked_x, axis=1)
else: 
    x = layers.GlobalMaxPooling()(x)

x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=2e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

/Users/orbinsunny/.pyenv/versions/3.12.3/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_encoder_4' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/orbinsunny/.pyenv/versions/3.12.3/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_encoder_5' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


### C. Implementing a pre-trained Transformer model for text classification.

### D.  Fine-tuning a pre-trained Transformer model on a custom dataset.

### E. Implementing optimization techniques in Transformer models.

## Practice

In [ ]:
import tensorflow as tf
from keras import layers
from tensorflow import keras 

# Data 
vocab_size = 20000
max_len = 200 

(x_train,y_train),(x_test, y_test) = keras.dataset.imdb.load_data(num_words=vocab_size) 
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len, padding='post')

# Position Encoding 
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = tf.range(max_len)[:, tf.newaxis]
        i = tf.range(d_model)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, )